# MAZZ ÆNGENIUM — FORGE TITAN 32B COT (Kaggle)
Ce notebook forge en ~6 minutes le modèle souverain de raisonnement **Mazz-Technos-Titan-32B-CoT** :
1. Base : DeepSeek-R1-Distill-Qwen-32B ablitérée (zéro censure, raisonnement Chain-of-Thought pur)
2. Quantification : `Q4_K_M` (18.49 Go)
3. Téléversement automatique vers votre dépôt privé Hugging Face `mazz92/mazz-models`

**Quota utilisé** : 100 % Gratuit (Kaggle Standard CPU ou GPU T4)

In [ ]:
# 1. Installation
!pip install -q huggingface_hub

In [ ]:
# 2. Forge et téléversement direct vers mazz92/mazz-models
import os, shutil
from huggingface_hub import hf_hub_download, HfApi

def _load_hf_token():
    import os
    try:
        from google.colab import userdata  # Colab : Secrets > HF_TOKEN
        return userdata.get('HF_TOKEN')
    except Exception:
        pass
    try:
        from kaggle_secrets import UserSecretClient  # Kaggle : Add-ons > Secrets > HF_TOKEN
        return UserSecretClient().get_secret('HF_TOKEN')
    except Exception:
        pass
    return os.environ.get('HF_TOKEN')

HF_TOKEN = _load_hf_token()
assert HF_TOKEN, 'HF_TOKEN absent : ajoutez-le dans les secrets Colab ou Kaggle'
REPO_ID = "mazz92/mazz-models"
TARGET_FILENAME = "Mazz-Technos-Titan-32B-CoT.Q4_K_M.gguf"

print("1/3 Téléchargement de la base DeepSeek-R1 32B ablitérée (18.49 Go)...", flush=True)
downloaded_path = hf_hub_download(
    repo_id="bartowski/DeepSeek-R1-Distill-Qwen-32B-abliterated-GGUF",
    filename="DeepSeek-R1-Distill-Qwen-32B-abliterated-Q4_K_M.gguf",
    local_dir="/kaggle/working"
)

titan_path = os.path.join("/kaggle/working", TARGET_FILENAME)
if os.path.exists(titan_path):
    os.remove(titan_path)
shutil.move(downloaded_path, titan_path)
sz_gb = os.path.getsize(titan_path) / (1024**3)
print(f"2/3 Modèle forgé : {TARGET_FILENAME} ({sz_gb:.2f} Go)", flush=True)

print(f"3/3 Téléversement vers votre dépôt privé Hugging Face {REPO_ID}...", flush=True)
api = HfApi(token=HF_TOKEN)
url = api.upload_file(
    path_or_fileobj=titan_path,
    path_in_repo=TARGET_FILENAME,
    repo_id=REPO_ID,
    repo_type="model"
)

print("=== SUCCÈS TOTAL TITAN 32B COT ===", flush=True)
print("Modèle disponible sur :", url)